# AI Full Body Video Generator (MimicMotion)

Ini adalah notebook Google Colab untuk menjalankan MimicMotion, sebuah sistem AI yang dapat menghasilkan video gerakan tubuh penuh berdasarkan foto target dan video referensi.

### Langkah 1: Persiapan Environment
Pastikan Anda menggunakan GPU (**Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**).

In [ ]:
# @title 1. Cek GPU
# Jalankan ini untuk memastikan GPU sudah aktif
!nvidia-smi

### Langkah 2: Instalasi Dependensi
Proses ini akan menginstal semua pustaka Python yang diperlukan. Kita menggunakan versi yang kompatibel dengan Python 3.12 di Colab.

In [ ]:
# @title 2. Instalasi Dependensi
# Update pip dan build tools untuk stabilitas
!pip install -U pip setuptools wheel

# Instalasi package utama
!apt-get update && apt-get install -y ffmpeg
!pip install diffusers>=0.27.0 transformers>=4.36.0 decord==0.6.0 einops omegaconf onnxruntime-gpu gradio accelerate imageio-ffmpeg av

**Note:** Jika Anda menemui error saat menginstal `tokenizers`, jalankan perintah di bawah ini untuk menginstal Rust compiler, lalu ulangi Langkah 2:
```bash
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] += ':' + os.path.expanduser('~/.cargo/bin')
```

### Langkah 3: Kloning Repositori
Mengambil kode sumber MimicMotion dari GitHub.

In [ ]:
# @title 3. Kloning Repositori MimicMotion
import os
if not os.path.exists('MimicMotion'):
    !git clone https://github.com/Tencent/MimicMotion.git
%cd MimicMotion

### Langkah 4: Download Model Weights
Mengunduh file 'otak' AI yang diperlukan (DWPose dan MimicMotion). Total sekitar 4-5 GB.

In [ ]:
# @title 4. Download Model Weights
!mkdir -p models/DWPose
!wget https://huggingface.co/yzd-v/DWPose/resolve/main/yolox_l.onnx -O models/DWPose/yolox_l.onnx
!wget https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.onnx -O models/DWPose/dw-ll_ucoco_384.onnx
!wget https://huggingface.co/tencent/MimicMotion/resolve/main/MimicMotion_1-1.pth -O models/MimicMotion_1-1.pth

### Langkah 5: Jalankan Aplikasi
Setelah dijalankan, tunggu hingga muncul link `public URL` yang berakhir dengan `.gradio.live`. Klik link tersebut untuk membuka antarmuka generator.

In [ ]:
# @title 5. Jalankan Aplikasi (Gradio)
app_code = r"""
import os
import math
import torch
import numpy as np
import gradio as gr
from pathlib import Path
from datetime import datetime
from omegaconf import OmegaConf
from PIL import Image
from torchvision.datasets.folder import pil_loader
from torchvision.transforms.functional import pil_to_tensor, resize, center_crop, to_pil_image

# MimicMotion imports
from mimicmotion.utils.geglu_patch import patch_geglu_inplace
patch_geglu_inplace()

from constants import ASPECT_RATIO
from mimicmotion.pipelines.pipeline_mimicmotion import MimicMotionPipeline
from mimicmotion.utils.loader import create_pipeline
from mimicmotion.utils.utils import save_to_mp4
from mimicmotion.dwpose.preprocess import get_video_pose, get_image_pose

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def preprocess(video_path, image_path, resolution=576, sample_stride=2):
    image_pixels = pil_loader(image_path)
    image_pixels = pil_to_tensor(image_pixels) # (c, h, w)
    h, w = image_pixels.shape[-2:]
    if h > w:
        w_target, h_target = resolution, int(resolution / ASPECT_RATIO // 64) * 64
    else:
        w_target, h_target = int(resolution / ASPECT_RATIO // 64) * 64, resolution
    h_w_ratio = float(h) / float(w)
    if h_w_ratio < h_target / w_target:
        h_resize, w_resize = h_target, math.ceil(h_target / h_w_ratio)
    else:
        h_resize, w_resize = math.ceil(w_target * h_w_ratio), w_target
    image_pixels = resize(image_pixels, [h_resize, w_resize], antialias=None)
    image_pixels = center_crop(image_pixels, [h_target, w_target])
    image_pixels = image_pixels.permute((1, 2, 0)).numpy()
    
    image_pose = get_image_pose(image_pixels)
    video_pose = get_video_pose(video_path, image_pixels, sample_stride=sample_stride)
    pose_pixels = np.concatenate([np.expand_dims(image_pose, 0), video_pose])
    image_pixels = np.transpose(np.expand_dims(image_pixels, 0), (0, 3, 1, 2))
    return torch.from_numpy(pose_pixels.copy()) / 127.5 - 1, torch.from_numpy(image_pixels) / 127.5 - 1

def run_pipeline(pipeline, image_pixels, pose_pixels, device, task_config):
    image_pixels = [to_pil_image(img.to(torch.uint8)) for img in (image_pixels + 1.0) * 127.5]
    generator = torch.Generator(device=device)
    generator.manual_seed(task_config.seed)
    frames = pipeline(
        image_pixels, image_pose=pose_pixels, num_frames=pose_pixels.size(0),
        tile_size=task_config.num_frames, tile_overlap=task_config.frames_overlap,
        height=pose_pixels.shape[-2], width=pose_pixels.shape[-1], fps=7,
        noise_aug_strength=task_config.noise_aug_strength, num_inference_steps=task_config.num_inference_steps,
        generator=generator, min_guidance_scale=task_config.guidance_scale, 
        max_guidance_scale=task_config.guidance_scale, decode_chunk_size=8, output_type="pt", device=device
    ).frames.cpu()
    video_frames = (frames * 255.0).to(torch.uint8)
    return video_frames[0, 1:]

pipeline = None

def load_model():
    global pipeline
    if pipeline is None:
        infer_config = OmegaConf.load("configs/test.yaml")
        pipeline = create_pipeline(infer_config, device)
    return pipeline

def generate_video(ref_image, ref_video, resolution, sample_stride, num_inference_steps, seed):
    global pipeline
    if pipeline is None:
        pipeline = load_model()
    
    torch.set_default_dtype(torch.float16)
    
    ref_image_path = "temp_image.png"
    ref_image.save(ref_image_path)
    ref_video_path = ref_video 

    task_config = OmegaConf.create({
        "ref_video_path": ref_video_path,
        "ref_image_path": ref_image_path,
        "resolution": resolution,
        "sample_stride": sample_stride,
        "num_frames": 72,
        "frames_overlap": 6,
        "noise_aug_strength": 0.02,
        "guidance_scale": 2.0,
        "num_inference_steps": num_inference_steps,
        "seed": seed,
        "fps": 15
    })

    pose_pixels, image_pixels = preprocess(
        task_config.ref_video_path, task_config.ref_image_path, 
        resolution=task_config.resolution, sample_stride=task_config.sample_stride
    )
    
    _video_frames = run_pipeline(pipeline, image_pixels, pose_pixels, device, task_config)
    
    output_path = f"outputs/output_{datetime.now().strftime('%Y%m%d%H%M%S')}.mp4"
    os.makedirs("outputs", exist_ok=True)
    save_to_mp4(_video_frames, output_path, fps=task_config.fps)
    
    return output_path

with gr.Blocks() as demo:
    gr.Markdown("# MimicMotion AI Full Body Video Generator")
    with gr.Row():
        with gr.Column():
            ref_image = gr.Image(label="Reference Image (Full Body)", type="pil")
            ref_video = gr.Video(label="Reference Video (Motion)")
            resolution = gr.Slider(minimum=256, maximum=1024, value=576, step=64, label="Resolution")
            sample_stride = gr.Slider(minimum=1, maximum=10, value=2, step=1, label="Sample Stride")
            num_inference_steps = gr.Slider(minimum=1, maximum=50, value=25, step=1, label="Inference Steps")
            seed = gr.Number(value=42, label="Seed")
            btn = gr.Button("Generate")
        with gr.Column():
            output_video = gr.Video(label="Generated Video")
    
    btn.click(
        generate_video, 
        inputs=[ref_image, ref_video, resolution, sample_stride, num_inference_steps, seed], 
        outputs=output_video
    )

if __name__ == "__main__":
    demo.launch(share=True)
"""
with open("app_colab.py", "w") as f:
    f.write(app_code)

!python app_colab.py